# LSTM 모델 구현하기

### 문제 설명
PyTorch에서 간단한 **LSTM(Long Short-Term Memory)** 모델을 구현합니다. 이 모델은 LSTM 레이어와 완전 연결(FC) 레이어를 통해 순차 데이터를 처리해야 합니다. 목표는 LSTM 레이어를 정의하고 forward pass를 구현하여 모델을 완성하는 것입니다.

### 요구사항
1. **LSTM 모델 정의**:
   - 모델에 **LSTM 레이어**를 추가합니다. 이 레이어는 입력 시퀀스를 받아 hidden state를 출력해야 합니다.
   - LSTM의 출력을 최종 예측값으로 매핑하는 **완전 연결(FC) 레이어**를 추가합니다.
   - `forward` 메서드를 구현해 다음을 수행합니다:
     - 입력 시퀀스를 LSTM에 통과시킵니다.
     - LSTM의 출력을 완전 연결 레이어에 전달해 최종 출력을 얻습니다.

### 제약 사항
- LSTM 레이어는 단일 hidden layer로 구현해야 합니다.
- 과제에 맞는 적절한 입력 feature 수, hidden unit 수, 출력 크기를 사용하세요.
- `forward` 메서드는 LSTM 출력을 처리한 뒤 완전 연결 레이어의 출력을 반환해야 합니다.

<details>
  <summary>💡 힌트</summary>
  `LSTMModel.__init__`에 LSTM 레이어와 FC 레이어를 추가하세요.
  <br>
  `forward` pass를 구현해 LSTM과 FC 레이어로 시퀀스를 처리하세요.
</details>


In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

In [2]:
# Generate synthetic sequential data
torch.manual_seed(42)
sequence_length = 10
num_samples = 100

# Create a sine wave dataset
X = torch.linspace(0, 4 * 3.14159, steps=num_samples).unsqueeze(1)
y = torch.sin(X)

# Prepare data for LSTM
def create_in_out_sequences(data, seq_length):
    in_seq = []
    out_seq = []
    for i in range(len(data) - seq_length):
        in_seq.append(data[i:i + seq_length])
        out_seq.append(data[i + seq_length])
    return torch.stack(in_seq), torch.stack(out_seq)

X_seq, y_seq = create_in_out_sequences(y, sequence_length)
X_seq.shape, y_seq.shape

(torch.Size([90, 10, 1]), torch.Size([90, 1]))

In [3]:
# Define the LSTM Model
# TODO: Add LSTM layer, forward implementation
class LSTMModel(nn.Module):
    # 스터디:
    # https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
    # 저는 마지막 시퀀스의 hidden state를 활용하지 않고 전체 시퀀스를 기반으로 사용해봤습니다

    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(1, 50, 1) # input_size, hidden_size, num_layers
        self.fc = nn.Linear(50, 1) # upproj 한거 다시 원래대로 돌려놓기
        self.out = nn.Linear(10, 1) # 시퀀스 downproj
    def forward(self, x):
        x, grad = self.lstm(x)
        x = self.fc(x) # 이거 거치면 (90, 10, 1) 이 됨. 각 시퀀스가 90개 차원인듯?
        x = torch.flatten(x, start_dim=1) # 마지막 1 차원 버리기
        x = self.out(x) # (90, 1)로 projection
        return x
# Initialize the model, loss function, and optimizer
model = LSTMModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [7]:
# Define the LSTM Model
# TODO: Add LSTM layer, forward implementation
class LSTMModel(nn.Module):
    # https://pytorch.org/docs/stable/generated/torch.nn.LSTM.html
    # 이건 last_hidden_state 활용한 방법

    def __init__(self):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(1, 50, 1, batch_first=True) # input_size, hidden_size, num_layers
        self.fc = nn.Linear(50, 1) # upproj 한거 다시 원래대로 돌려놓기
    def forward(self, x):
        x, shape = self.lstm(x)
        x = x[:, -1, :]
        x = self.fc(x)
        return x
# Initialize the model, loss function, and optimizer
model = LSTMModel()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [5]:
prediction = model(X_seq)
prediction.shape

torch.Size([90, 1])

In [8]:
# Training loop
epochs = 500
for epoch in range(epochs):
    # Forward pass
    predictions = model(X_seq)
    loss = criterion(predictions, y_seq)

    # Backward pass and optimization
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    # Log progress every 50 epochs
    if (epoch + 1) % 50 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Loss: {loss.item():.4f}")

Epoch [50/500], Loss: 0.0008
Epoch [100/500], Loss: 0.0000
Epoch [150/500], Loss: 0.0000
Epoch [200/500], Loss: 0.0000
Epoch [250/500], Loss: 0.0000
Epoch [300/500], Loss: 0.0000
Epoch [350/500], Loss: 0.0000
Epoch [400/500], Loss: 0.0000
Epoch [450/500], Loss: 0.0000
Epoch [500/500], Loss: 0.0001


로스 0.001 미만으로 찍히면 통과하겠습니다. RNN이 완전 0을 만들 수 있긴 한데 굳이?

In [9]:
# Testing on new data
test_steps = 20  # Ensure this is greater than sequence_length
X_test = torch.linspace(4 * 3.14159, 5 * 3.14159, steps=test_steps).unsqueeze(1)
y_test = torch.sin(X_test)

# Create test input sequences
X_test_seq, _ = create_in_out_sequences(y_test, sequence_length)

with torch.no_grad():
    predictions = model(X_test_seq)
    print(f"Predictions for new sequence: {predictions.squeeze().tolist()}")


Predictions for new sequence: [1.0141814947128296, 0.9903647899627686, 0.9388716220855713, 0.8606303930282593, 0.7574792504310608, 0.6322651505470276, 0.4888419508934021, 0.33186963200569153, 0.16640979051589966, -0.0025227442383766174]
